# LTSM

based on https://machinelearningmastery.com/text-generation-lstm-recurrent-neural-networks-python-keras/

In [27]:
import sys
#!conda install --yes --prefix {sys.prefix} tensorflow

In [28]:
# load packages
import numpy
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import LSTM
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import to_categorical
#from tensorflow.keras.utils import np_utils

In [29]:
# load text and covert to lowercase
filename = "Bowie.txt"
raw_text = open(filename, 'r', encoding='utf-8').read()
raw_text = raw_text.lower()

In [30]:
# create mapping of unique chars to integers, and a reverse mapping
chars = sorted(list(set(raw_text)))
char_to_int = dict((c, i) for i, c in enumerate(chars))
int_to_char = dict((i, c) for i, c in enumerate(chars))

In [31]:
int_to_char

{0: '\n',
 1: ' ',
 2: '!',
 3: '"',
 4: '&',
 5: "'",
 6: '(',
 7: ')',
 8: '*',
 9: ',',
 10: '-',
 11: '.',
 12: '/',
 13: '0',
 14: '1',
 15: '2',
 16: '3',
 17: '4',
 18: '5',
 19: '6',
 20: '7',
 21: '8',
 22: '9',
 23: ':',
 24: ';',
 25: '?',
 26: '[',
 27: ']',
 28: '_',
 29: 'a',
 30: 'b',
 31: 'c',
 32: 'd',
 33: 'e',
 34: 'f',
 35: 'g',
 36: 'h',
 37: 'i',
 38: 'j',
 39: 'k',
 40: 'l',
 41: 'm',
 42: 'n',
 43: 'o',
 44: 'p',
 45: 'q',
 46: 'r',
 47: 's',
 48: 't',
 49: 'u',
 50: 'v',
 51: 'w',
 52: 'x',
 53: 'y',
 54: 'z',
 55: '{',
 56: '}',
 57: 'ß',
 58: 'à',
 59: 'ä',
 60: 'è',
 61: 'é',
 62: 'ê',
 63: 'ñ',
 64: 'ò',
 65: 'ô',
 66: 'ö',
 67: 'ù',
 68: 'ü',
 69: '‘',
 70: '’'}

In [32]:
# summarize the loaded data
n_chars = len(raw_text)
n_vocab = len(chars)
print("Total Characters: ", n_chars)
print("Total Vocab (Different characters): ", n_vocab)
# prepare the dataset of input to output pairs encoded as integers
seq_length = 100
dataX = []
dataY = []
for i in range(0, n_chars - seq_length, 1):
    seq_in = raw_text[i:i + seq_length]
    seq_out = raw_text[i + seq_length]
    dataX.append([char_to_int[char] for char in seq_in])
    dataY.append(char_to_int[seq_out])
n_patterns = len(dataX)
print("Total Patterns: ", n_patterns)




Total Characters:  950734
Total Vocab (Different characters):  71
Total Patterns:  950634


In [33]:
dataX[0]

[43,
 36,
 1,
 44,
 37,
 48,
 53,
 1,
 49,
 47,
 1,
 36,
 33,
 46,
 33,
 1,
 51,
 33,
 1,
 29,
 42,
 35,
 33,
 40,
 47,
 1,
 43,
 34,
 1,
 40,
 33,
 29,
 32,
 0,
 51,
 33,
 5,
 46,
 33,
 1,
 32,
 33,
 29,
 32,
 9,
 1,
 51,
 33,
 5,
 46,
 33,
 1,
 47,
 37,
 31,
 39,
 1,
 36,
 29,
 42,
 35,
 37,
 42,
 35,
 1,
 30,
 53,
 1,
 48,
 36,
 46,
 33,
 29,
 32,
 0,
 35,
 33,
 48,
 1,
 46,
 33,
 29,
 40,
 0,
 35,
 33,
 48,
 1,
 46,
 33,
 29,
 40,
 0,
 53,
 43,
 49,
 1,
 31,
 29,
 42]

In [34]:

# reshape X to be [samples, time steps, features]
X = numpy.reshape(dataX, (n_patterns, seq_length, 1))
# normalize
X = X / float(n_vocab)
# one hot encode the output variable
y = to_categorical(dataY)

In [35]:
X.shape

(950634, 100, 1)

In [36]:
# define the LSTM model with Keras
model = Sequential()
model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2])))
model.add(Dropout(0.2))
model.add(Dense(100, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(y.shape[1], activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')
# define the checkpoint
filepath="weights-improvement3-{epoch:02d}-{loss:.4f}.keras"
checkpoint = ModelCheckpoint(filepath, monitor='loss', verbose=1, save_best_only=True, mode='min')
callbacks_list = [checkpoint]

In [37]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_3 (LSTM)                   │ (None, 256)            │       264,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 100)            │        25,700 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 71)             │         7,171 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,063 (1.13 MB)

 Trainable params: 297,063 (1.13 MB)

 Non-trainable params: 0 (0.00 B)

In [38]:
# ALERT: very computationally expensive!

# load the network weights
filename = "./weights-improvement3-13-1.8229.keras"
model.load_weights(filename)

#model.compile(loss='categorical_crossentropy', optimizer='adam')
# fit the model
#model.fit(X, y, epochs=100, batch_size=300, callbacks=callbacks_list)

In [39]:
# pick a random seed
# start = numpy.random.randint(0, len(dataX)-1)
# pattern = dataX[start]
# print("Seed:")
# print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")
# # generate characters
# for i in range(1000):
#     x = numpy.reshape(pattern, (1, len(pattern), 1))
#     x = x / float(n_vocab)
#     prediction = model.predict(x, verbose=0)
#     index = numpy.argmax(prediction)
#     result = int_to_char[index]
#     seq_in = [int_to_char[value] for value in pattern]
#     sys.stdout.write(result)
#     pattern.append(index)
#     pattern = pattern[1:len(pattern)]
# print("\nDone.")

In [45]:
# # define the LSTM model with Keras
# model = Sequential()
# model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2])))
# model.add(Dropout(0.2))
# model.add(Dense(100, activation='relu'))
# model.add(Dropout(0.2))
# model.add(Dense(y.shape[1], activation='softmax'))

# # load the network weights
# filename = "./weights-improvement3-13-1.8229.keras"
# model.load_weights(filename)
# model.compile(loss='categorical_crossentropy', optimizer='adam')

# pick a random seed
start = numpy.random.randint(0, len(dataX)-1)
pattern = dataX[start]
print("Seed:")
print("\"", ''.join([int_to_char[value] for value in pattern]), "\"")
print("---------")

# generate characters
for i in range(200):
    x = numpy.reshape(pattern, (1, len(pattern), 1))
    x = x / float(n_vocab)
    prediction = model.predict(x, verbose=0)
    index = numpy.argmax(prediction)
    result = int_to_char[index]
    seq_in = [int_to_char[value] for value in pattern]
    sys.stdout.write(result)
    pattern.append(index)
    pattern = pattern[1:len(pattern)]
print("\nDone.")

Seed:
" ound while shaking in fright
i guess we could cruise down one more time
with you by my side, it shou "
---------
 so the sornd of the saie 
she sail the saie 
she sail the say she saie she saie she say she saie she say she saie she say she saie she say she saie she saie she say she saie she saie she say she saie
Done.
